# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoussifKhaled77/FlyrankAI-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Ranking & Binary Classification (Scoring)

**Why:**
This task is framed as a **ranking and binary classification problem**. While the underlying model predicts a continuous risk score (probability between 0 and 1 that a page requires a content refresh), the primary business deliverable is a **ranked queue** sorted by risk score.

Binary classification alone is insufficient because reviewing team capacity is strictly limited (e.g., top 20–50 items per cycle out of 30,000 total pages). Therefore, the raw probability must serve as a ranking metric to prioritize high-value, high-risk pages at the top of the reviewer's queue.

In [ ]:
import pandas as pd

# Load dataset and check dimensions
df = pd.read_csv("/content/content_refresh_anonymized.csv")

print(f"Rows x columns: {df.shape[0]:,} x {df.shape[1]}")
print(f"Distinct clients (client_id): {df['client_id'].nunique()}")
print(f"Distinct content items (content_id): {df['content_id'].nunique()}")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / Proxy Definition:**
The target variable is `is_declining_label`, a binary proxy derived from a defined outcome rule: `trend_direction == 'down'`.

**Label Origin & Leakage Boundary:**
* **Derived Rule (Proxy):** The label is constructed by comparing performance metrics (e.g., clicks/impressions) over the current 30-day window against the prior 30-day window.
* **Leakage Caution:** Because `trend_direction` and `trend_pct` directly define the target label, these columns are strictly isolated as label sources and will **never** be used as input features for model training.

In [5]:
# Verify label source columns and confirm unit of analysis uniqueness
label_source_columns = ["trend_direction", "trend_pct"]
print("Label-source columns (isolated, never used as features):", label_source_columns)

print(f"content_id is unique per row: {df['content_id'].is_unique}")
print(f"client_id distinct pseudonymous values: {df['client_id'].nunique()}")

Label-source columns (isolated, never used as features): ['trend_direction', 'trend_pct']
content_id is unique per row: True
client_id distinct pseudonymous values: 32


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Metric:** Precision@50

**Defendable Threshold:** Precision@50 >= 0.70 (70% or higher of the top 50 surfaced items are true positives requiring refresh).

**Why:**
Human review capacity is the true operational bottleneck; reviewers only have time to evaluate a small batch of pages (e.g., top 50) per cycle. Maximizing Precision@50 minimizes wasted reviewer effort on false positives (such as seasonal drops or temporary fluctuations), ensuring that limited reviewer hours are spent on high-impact content.

In [6]:
# Measure the scale of demand associated with declining pages
declining_with_demand = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)

n_flagged = declining_with_demand.sum()
pct_flagged = declining_with_demand.mean() * 100
impressions_at_stake = df.loc[declining_with_demand, "impressions_90d"].sum()
impressions_total = df["impressions_90d"].sum()
pct_impressions_at_stake = 100 * impressions_at_stake / impressions_total

print(f"Pages trending down with demand (impressions_90d >= 100): {n_flagged:,} ({pct_flagged:.1f}% of total)")
print(f"Share of total impressions on declining pages: {pct_impressions_at_stake:.1f}%")

Pages trending down with demand (impressions_90d >= 100): 13,152 (43.8% of total)
Share of total impressions on declining pages: 51.2%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** One row = One unique content item (`content_id`) for a specific client (`client_id`) evaluated over a 90-day observation window.

In [9]:
# Check exact column names in your dataframe
print([c for c in df.columns if 'position' in c or '90d' in c])

['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'avg_position', 'position_tier']


In [10]:
# Display the primary key and feature representation slice of the unit of analysis
unit_of_analysis_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "trend_direction",
]

df[unit_of_analysis_cols].head()

,content_id,client_id,impressions_90d,clicks_90d,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,down
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML Beats Fixed Rules:**
A simple heuristic rule (e.g., `impressions_90d >= 100` AND `trend_direction == 'down'`) flags over 13,000 pages—far too large a backlog for manual review. Furthermore, basic heuristic rules reach low precision (Precision@50 approx. 0.24).

Content risk involves complex, non-linear interactions across continuous features (impressions, clicks, average position, engagement metrics, and freshness). A machine learning model (e.g., Random Forest or Gradient Boosting) learns these non-linear interactions to boost Precision@50 to approx. 0.74—roughly 3x the precision of a naive fixed rule.

In [11]:
# Summary statistics demonstrating why rules create huge backlogs requiring ML ranking
trend_counts = df["trend_direction"].value_counts()
trend_pct = (df["trend_direction"].value_counts(normalize=True) * 100).round(1)

print("Distribution of trend_direction across dataset:")
print(pd.DataFrame({"Count": trend_counts, "Percentage (%)": trend_pct}))

Distribution of trend_direction across dataset:
                 Count  Percentage (%)
trend_direction                       
down             16262            54.2
stable            5962            19.9
up                4388            14.6
new               2236             7.5
flat              1152             3.8


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.